<a href="https://colab.research.google.com/github/selooo0/gulf-ticket-assistant/blob/main/ticket_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# مساعد تذاكر خدمة العملاء للهجة الخليجية
### Gulf Arabic Customer-Support Ticket Assistant

مشروع ختامي: **SDAIA Academy · Applied NLP Engineering** ([github.com/SDAIAAcademy](https://github.com/SDAIAAcademy))

**المشكلة:** المتاجر السعودية تستقبل رسائل عملاء باللهجة الخليجية، والنماذج المدربة على الإنجليزي والفصحى تفشل فيها. النموذج الأساسي يصل 0.99 على الإنجليزي لكن 0.40 فقط على الخليجي.

**المدخل:** رسالة عميل نصية (إنجليزي، فصحى، خليجي).

**المخرج:** نوع الطلب (intent) + درجة الثقة + المسار (تلقائي أو تحويل لموظف) + الأولوية + رقم الطلب + أقرب 3 تذاكر سابقة + رد مقترح بالعربي.

**النطاق:** 8 أنواع طلبات: `get_refund, track_refund, payment_issue, cancel_order, track_order, recover_password, change_shipping_address, complaint`

**مقاييس النجاح:** Macro-F1 على اختبار خليجي مكتوب يدوياً ≥ 0.80، الإنجليزي ما ينزل عن 0.99، ونسبة اتفاق نموذج الحكم مع التقييم البشري ≥ 80%.

> **ملاحظة:** كل الخطوات اللي تستخدم LLM تحفظ نتائجها في Drive. تشغيل **Run all** مرة ثانية ما يصرف أي طلب API، لأنه يقرأ النتائج المحفوظة.

## 1. الإعداد

In [56]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/applied-nlp.git"
    if not Path("/content/applied-nlp").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/applied-nlp"], check=True)
    os.chdir("/content/applied-nlp/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
if not Path("data/tickets_train.csv").exists():
    subprocess.run([sys.executable, "prepare_data.py"], check=True)
sys.path.insert(0, os.getcwd())

import nlp_utils as U
U.set_seed(42)
print("device:", U.device(), "| quick mode:", U.QUICK)

device: cpu | quick mode: False


In [57]:
import re, json, time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, cohen_kappa_score
from google.colab import userdata
REPO = "/content/gulf-ticket-assistant"
if not Path(REPO).exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/selooo0/gulf-ticket-assistant.git", REPO], check=True)
PROJ = f"{REPO}/data"
pd.set_option("display.max_colwidth", 120)
print("ملفات المشروع:", sorted(os.listdir(PROJ)))

ملفات المشروع: ['NLP_Intent_Classification.csv', 'evaluation.csv', 'generated_replies.csv', 'gulf_train.csv', 'gulf_train_split.csv', 'gulf_val.csv', 'human_vs_judge.csv', 'judge_labels.csv', 'manual_labels.csv', '\u200f\u200fevaluation - نسخة.csv']


## 2. البيانات

| الجزء | المصدر | العدد | الاستخدام |
|---|---|---|---|
| Train | Bitext (إنجليزي، 27 نوع) + خليجي مولّد ومراجَع | آلاف + 147 | التدريب |
| Validation | خليجي مولّد ومراجَع | 40 | اختيار النموذج والحد τ |
| Test | **خليجي مكتوب يدوياً** (8 أنواع × 5) | 40 | القياس النهائي فقط |
| History | ردود موظفين سابقة (Bitext) | آلاف | قاعدة البحث (RAG) |

بيانات التدريب الخليجية (190 جملة) مولّدة، وراجعت منها 32 جملة يدوياً (97% صحيحة). بيانات الاختبار كتبتها يدوياً ولا تدخل في التدريب أبداً.

In [58]:
train, val, test = U.load_tickets("train"), U.load_tickets("val"), U.load_tickets("test")
history = U.load_tickets("history")
if U.QUICK:
    history = history.sample(1000, random_state=0).sort_index().reset_index(drop=True)

gulf   = pd.read_csv(f"{PROJ}/NLP_Intent_Classification.csv")   # اختبار خليجي يدوي (40)
gtrain = pd.read_csv(f"{PROJ}/gulf_train.csv")                  # خليجي مولّد (190)
L = sorted(gulf.intent.unique())

g_tr, g_val = train_test_split(gtrain, test_size=40, stratify=gtrain.intent, random_state=42)
print(f"EN train: {len(train)} | EN test: {len(test)} | history: {len(history)}")
print(f"Gulf train: {len(g_tr)} | Gulf val: {len(g_val)} | Gulf test: {len(gulf)}")

EN train: 5400 | EN test: 1080 | history: 8000
Gulf train: 150 | Gulf val: 40 | Gulf test: 40


### فحص التسريب (Leakage)
نتأكد إن ما فيه جملة اختبار شبه مطابقة لجملة تدريب (تشابه > 0.95).

In [59]:
ALT = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
E_tr = U.embed(g_tr.text, prefix="", model=ALT)
E_te = U.embed(gulf.text, prefix="", model=ALT)
S = E_te @ E_tr.T
dups = pd.DataFrame({"test": gulf.text, "train_match": g_tr.text.values[S.argmax(1)], "sim": S.max(1).round(3)})
display(dups[dups.sim > 0.95])

keep = (E_tr @ E_te.T).max(axis=1) <= 0.95
g_tr = g_tr[keep].reset_index(drop=True)
print("Gulf train بعد حذف الجمل الشبيهة:", len(g_tr))

,test,train_match,sim
8,ابغى اعرف وش صار على فلوسي اللي طلبت استرجاعها,كلمة المرور ما ادري وش هي,0.981
13,ليه كل ما جيت ادفع يرفض العمليه مع ان رصيدي موجود؟!,الفلوس اللي استرجعتها وين راحت؟,0.952


Gulf train بعد حذف الجمل الشبيهة: 147


**النتيجة:** جملتين اختبار طلعت تشابهها > 0.95، لكن بالمراجعة اليدوية معانيها مختلفة تماماً (فلوس مقابل كلمة مرور). يعني ما فيه تسريب حقيقي، والنموذج نفسه ما يفهم الخليجي بدقة. حذفنا الجمل الشبيهة من التدريب احتياطاً.

## اكتشاف المشكلة
نختبر نموذجين جاهزين (مدربين على الإنجليزي فقط) على اختبار الكورس العربي الموجود بالتدريب ، اللي فيه **فصحى** و**خليجي**  .

In [60]:
def LR(): return LogisticRegression(max_iter=2000, C=4.0)
ar = U.load_tickets("ar_test")

X_tr  = U.embed(train.text, prefix="query: ", cache_key="train")
X_te  = U.embed(test.text,  prefix="query: ", cache_key="test")
X_ar  = U.embed(ar.text,    prefix="query: ", cache_key="ar_test")
X_tr2 = U.embed(train.text, prefix="", model=ALT, cache_key="train")
X_te2 = U.embed(test.text,  prefix="", model=ALT, cache_key="test")
X_ar2 = U.embed(ar.text,    prefix="", model=ALT, cache_key="ar_test")

e5   = LR().fit(X_tr, train.intent)
mini = LR().fit(X_tr2, train.intent)

disc = pd.DataFrame({"dialect": ar.dialect.values, "y": ar.intent.values,
                     "e5": e5.predict(X_ar), "MiniLM": mini.predict(X_ar2)})
rows = []
for name, m, X, col in [("e5 + LR", e5, X_te, "e5"), ("MiniLM + LR", mini, X_te2, "MiniLM")]:
    row = {"model": name, "English": f1_score(test.intent, m.predict(X), average="macro")}
    for d, g in disc.groupby("dialect"):
        row[d] = f1_score(g.y, g[col], average="macro")
    rows.append(row)
pd.DataFrame(rows).set_index("model").round(3)

,English,gulf,msa
model,,,
e5 + LR,0.994,0.307,0.624
MiniLM + LR,0.994,0.307,0.843


**المشكلة بالأرقام:** النموذج ممتاز على الإنجليزي (0.99) وجيد على الفصحى (حتى 0.84)، لكنه **ينهار على الخليجي (0.31)** في النموذجين. تغيير نموذج الـ embeddings حسّن الفصحى بس، **وما حل الخليجي**. فالمشكلة أعمق: النموذج ما شاف لهجة خليجية في التدريب.

وجربنا جمل خليجية من عندنا: كلها راحت `human_review` بثقة ضعيفة، وبعضها تصنيفها غلط تماماً (مثل "نسيت الباسوورد" ← `get_invoice`).

**هدف المشروع:** تقليص فجوة الخليجي، بدون ما يخرب الإنجليزي.

## 3. الـ Baselines واختيار النموذج
نقارن على **نفس** الاختبار الخليجي: TF-IDF، ثم نموذجين embeddings (e5 و MiniLM) مع Logistic Regression، قبل وبعد إضافة البيانات الخليجية.

In [61]:
def f1_gulf(y, p): return f1_score(y, p, labels=L, average="macro")
def f1_en(y, p):   return f1_score(y, p, average="macro")
def LR():          return LogisticRegression(max_iter=2000, C=4.0)

# embeddings
X_tr  = U.embed(train.text, prefix="query: ", cache_key="train")
X_te  = U.embed(test.text,  prefix="query: ", cache_key="test")
X_tr2 = U.embed(train.text, prefix="", model=ALT, cache_key="train")
X_te2 = U.embed(test.text,  prefix="", model=ALT, cache_key="test")
G5, V5, T5 = (U.embed(d.text, prefix="query: ") for d in (gulf, g_val, g_tr))
G2, V2, T2 = (U.embed(d.text, prefix="", model=ALT) for d in (gulf, g_val, g_tr))
y_aug = np.concatenate([train.intent.values, g_tr.intent.values])

tfidf  = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                       LogisticRegression(max_iter=2000, C=10.0)).fit(train.text, train.intent)
e5     = LR().fit(X_tr, train.intent)
mini   = LR().fit(X_tr2, train.intent)
e5_g   = LR().fit(np.vstack([X_tr, T5]), y_aug)
mini_g = LR().fit(np.vstack([X_tr2, T2]), y_aug)

rows = [{"model": "TF-IDF + LR (baseline)", "Gulf test": f1_gulf(gulf.intent, tfidf.predict(gulf.text)),
         "Gulf val": f1_gulf(g_val.intent, tfidf.predict(g_val.text)), "EN test": f1_en(test.intent, tfidf.predict(test.text))}]
for name, m, G, V, X in [("e5 + LR", e5, G5, V5, X_te), ("MiniLM + LR", mini, G2, V2, X_te2),
                         ("MiniLM + LR + Gulf", mini_g, G2, V2, X_te2), ("e5 + LR + Gulf (المختار)", e5_g, G5, V5, X_te)]:
    rows.append({"model": name, "Gulf test": f1_gulf(gulf.intent, m.predict(G)),
                 "Gulf val": f1_gulf(g_val.intent, m.predict(V)), "EN test": f1_en(test.intent, m.predict(X))})
results = pd.DataFrame(rows).set_index("model").round(3)
results

,Gulf test,Gulf val,EN test
model,,,
TF-IDF + LR (baseline),0.000,0.000,0.990
e5 + LR,0.265,0.292,0.994
MiniLM + LR,0.396,0.274,0.994
MiniLM + LR + Gulf,0.776,0.668,0.994
e5 + LR + Gulf (المختار),0.791,0.734,0.995


**اختيار النموذج:** بدون بيانات خليجية MiniLM كان أفضل من e5 على الخليجي، لكن مع البيانات الخليجية **e5 تفوّق بوضوح على الـ validation**، فاخترناه. القرار مبني على الـ validation وليس الـ test. والإنجليزي ما تأثر.

TF-IDF = صفر على الخليجي لأنه تدرّب على كلمات إنجليزية فقط، فكل كلمة عربية عنده مجهولة. النموذج الافتراضي في الكورس (e5) مو بالضرورة الأفضل لبياناتنا، فقسنا واخترنا بالأرقام.

⚠️ 40 جملة اختبار عدد صغير، فالفروقات الصغيرة (2 إلى 3 نقاط) ممكن تكون صدفة.

## 4. حد الثقة τ (متى يتحول للموظف؟)
الحد الافتراضي من اللاب (0.5) اختير على الإنجليزي. نختاره من جديد على الـ validation الخليجي.

In [62]:
clf = e5_g
P = clf.predict_proba(V5)
conf_v, correct_v = P.max(1), clf.classes_[P.argmax(1)] == g_val.intent.values
tau_rows = [{"tau": t, "تلقائي (coverage)": (conf_v >= t).mean(),
             "دقة التلقائي": correct_v[conf_v >= t].mean() if (conf_v >= t).any() else np.nan}
            for t in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]]
pd.DataFrame(tau_rows).round(2)

,tau,تلقائي (coverage),دقة التلقائي
0,0.10,1.00,0.75
1,0.15,1.00,0.75
2,0.20,0.82,0.82
3,0.25,0.62,0.88
4,0.30,0.55,0.91
5,0.40,0.32,1.00
6,0.50,0.18,1.00


In [63]:
TAU = 0.30   # مختار على الـ validation: 55% تلقائي بدقة 91%

**اختيار τ = 0.30:** يحل 55% من التذاكر تلقائياً بدقة 91%، والباقي يتحول لموظف. الحد 0.5 كان يحل 18% فقط، لأن المصنّف يوزع الثقة على 27 نوع.

## 5. البحث (RAG): أقرب ردود موظفين سابقة
**قبل:** البحث في كل التذاكر السابقة. **بعد:** البحث فقط في التذاكر من نفس النوع اللي توقعه المصنّف.

In [64]:
hist_emb = U.embed(history.text, prefix="passage: ", cache_key="history")

def top_idx(text, k=3, filtered=False):
    q = U.embed([text], prefix="query: ")[0]
    sims = hist_emb @ q
    if filtered:
        pred = clf.predict(q[None])[0]
        sims = np.where(history.intent.values == pred, sims, -1)
    return np.argsort(-sims)[:k]

rows = []
for t, y in zip(gulf.text, gulf.intent):
    for name, f in [("قبل (كل التذاكر)", False), ("بعد (نفس النوع المتوقع)", True)]:
        n = int((history.intent.values[top_idx(t, filtered=f)] == y).sum())
        rows.append({"method": name, "n_relevant": n, "hit": n > 0, "all3": n == 3})
r = pd.DataFrame(rows).groupby("method")[["hit", "all3", "n_relevant"]].mean().round(2)
r.columns = ["سياق صحيح واحد على الأقل", "الثلاثة كلها صحيحة", "متوسط الصحيح من 3"]
r

,سياق صحيح واحد على الأقل,الثلاثة كلها صحيحة,متوسط الصحيح من 3
method,,,
بعد (نفس النوع المتوقع),0.78,0.78,2.33
قبل (كل التذاكر),0.42,0.18,0.90


**النتيجة:** نسبة التذاكر اللي سياقها الثلاثة كلها صحيحة قفزت من 18% إلى 78%، بدون أي تكلفة.

**التوازن:** البحث صار يعتمد كلياً على المصنّف. إذا المصنّف غلط (22%)، السياق كله يكون غلط. حل مستقبلي: تذكرتين من النوع المتوقع + تذكرة من البحث العام.

## 6. توليد الردود (LLM)
- **نموذج التوليد:** `nvidia/nemotron-3-ultra-550b-a55b:free` (NVIDIA) عبر OpenRouter.
- **نموذج الحكم:** `dots-studio/dots-3-note-preview:free` (Dots Studio)، من شركة مختلفة عشان ما يمدح نفسه.
- اخترناهم من النماذج المجانية الأكثر استخداماً في OpenRouter، واختبرناهم على العربي. Inkling رُفض لأن نسخته المجانية غير متاحة للاستخدام المباشر.
- المفتاح محفوظ في Colab Secrets باسم `OPENROUTER_API_KEY` (مو في الكود).

ولّدنا ردود لـ 32 تذكرة (4 من كل نوع) من الاختبار الخليجي. **ملاحظة:** هذي الردود استخدمت البحث **قبل** التحسين.

In [65]:
from openai import OpenAI
try:
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))
except Exception:
    client = None
    print("ما فيه مفتاح API: النتائج المحفوظة تُقرأ من data/ والواجهة تعطي الرد الثابت")

GEN_MODEL   = "nvidia/nemotron-3-ultra-550b-a55b:free"
JUDGE_MODEL = "dots-studio/dots-3-note-preview:free"

SYSTEM = """أنت موظف خدمة عملاء في متجر إلكتروني سعودي.
اكتب رداً واحداً فقط للعميل بالعربي، من جملتين إلى ثلاث جمل، بأسلوب مهذب وواضح.
استخدم فقط المعلومات الموجودة في "ردود سابقة"، ولا تخترع مدداً أو أرقاماً أو سياسات غير مذكورة.
إذا ما كفت المعلومات، قل إن أحد الزملاء بيتابع معه.
لا تكتب خيارات متعددة ولا ملاحظات، فقط نص الرد."""

def build_context(idx):
    return "\n\n".join(f"[{i+1}] {history.response[j][:400]}" for i, j in enumerate(idx))

def generate(text, intent, context, tries=2):
    if client is None:
        return "شكراً لتواصلك معنا، تم استلام طلبك وبيتواصل معك أحد الزملاء بأقرب وقت."
    for _ in range(tries):
        r = client.chat.completions.create(
            model=GEN_MODEL, temperature=0.3, max_tokens=1500,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": f"نوع الطلب: {intent}\n\nردود سابقة لموظفين على طلبات مشابهة:\n{context}\n\nرسالة العميل:\n<ticket>{text}</ticket>"}])
        if r.choices and r.choices[0].message.content:
            return r.choices[0].message.content
        time.sleep(20)
    raise RuntimeError("النموذج ما رد بعد محاولتين")

In [66]:
OUT = f"{PROJ}/generated_replies.csv"
gen_set = gulf.groupby("intent").head(4).reset_index(drop=True)   # 32 تذكرة
rows = pd.read_csv(OUT).to_dict("records") if os.path.exists(OUT) else []
done = {r["text"] for r in rows}

for t, y in zip(gen_set.text, gen_set.intent):
    if t in done:
        continue
    pred = clf.predict(U.embed([t], prefix="query: "))[0]
    ctx = build_context(top_idx(t))            # البحث قبل التحسين
    try:
        reply = generate(t, pred, ctx)
    except Exception as e:
        print("❌ توقف:", str(e)[:200]); break
    rows.append({"text": t, "intent": y, "pred_intent": pred, "context": ctx, "reply": reply})
    pd.DataFrame(rows).to_csv(OUT, index=False)
    time.sleep(4)

replies = pd.DataFrame(rows)
print("عدد الردود:", len(replies))
replies[["text", "intent", "pred_intent", "reply"]].head(10)

عدد الردود: 32


,text,intent,pred_intent,reply
0,أبي أرجع الطلب واسترد فلوسى كيف الطريقة؟,get_refund,get_refund,أهلاً بك، سأطلب من أحد الزملاء متابعة طلبك لتوضيح خطوات الإرجاع والاسترداد بالتفصيل.
1,المنتج وصلني خربان وأبي فلوسي ترجع لي بأسرع وقت!,get_refund,delivery_period,أعتذر عن تجربتك السيئة، أحد الزملاء بيتابع معك بخصوص استرجاع المبلغ للمنتج التالف.
2,لو سمحتو كيف اقدر استرجع المبلغ حق الطلب؟,get_refund,get_refund,لاسترجاع المبلغ، يرجى تزويدنا برقم الطلب وتاريخ الشراء، ثم التواصل مع فريق خدمة العملاء عبر الموقع أو رقم الدعم المخ...
3,معليش ما ناسبني المنتج.ممكن أسترجعه واخذ المبلغ؟,get_refund,get_refund,أهلاً بك، أحد الزملاء بيتابع معك إجراءات الاسترجاع واسترداد المبلغ بالتفصيل.
4,وين وصل استرجاع فلوسي؟ لي أسبوع أنتظر!!!!!!!!,track_refund,track_refund,أعتذر عن التأخير في استرجاع المبلغ، حيث تختلف مدة المعالجة حسب طريقة الدفع والظروف المحيطة بالطلب. هل يمكنك تزويدي ب...
5,تم قبول طلب الاسترجاع بس للحين المبلغ ما نزل بحسابي.,track_refund,track_refund,أعتذر عن التأخير في ظهور المبلغ، أحد الزملاء سيتابع معكم حالة التحويل البنكي للتأكد من وصوله لحسابكم في أقرب وقت.
6,متى بترجعون المبلغ؟ صار لي كم يوم انتظر وعصبت مره,track_refund,track_refund,أعتذر عن التأخير وأتفهم استياءك، فإجراءات استرداد المبلغ تختلف حسب طريقة الدفع المستخدمة وأي عمليات تحقق مطلوبة. لتق...
7,ابغى اعرف وش صار على فلوسي اللي طلبت استرجاعها,track_refund,track_refund,أهلاً بك، أحتاج لبعض التفاصيل الإضافية للتحقق من حالة طلب الاسترجاع الخاص بك. أحد الزملاء المختصين بيتابع معك مباشرة...
8,حاولت ادفع بالبطاقة بس كل شوي يطلع لي خطأ!,payment_issue,payment_issue,أعتذر عن مشكلة الدفع التي تواجهها، أحد الزملاء بيتابع معك لحل مشكلة خطأ البطاقة.
9,انخصم المبلغ من حسابي والطلب ما تأكد، وش الحل؟,payment_issue,payment_issue,نعتذر عن هذا الخطأ، يمكنك طلب استرداد المبلغ بالتواصل مع فريق خدمة العملاء وتزويدهم بتفاصيل العملية والمستندات الداع...


## 7. التقييم اليدوي (بشري)
قيّمت كل رد بسؤالين (1/0):
1. **يجاوب الطلب:** هل الرد أفاد العميل؟ (التحويل لزميل = 1 إذا السياق ما فيه جواب، و 0 إذا الجواب كان موجود وتجاهله)
2. **بدون اختراع:** هل كل معلومة محددة في الرد (خطوات، روابط، مدد، وعود) موجودة في السياق؟

In [67]:
from IPython.display import clear_output
LBL = f"{PROJ}/manual_labels.csv"
labels_done = pd.read_csv(LBL).to_dict("records") if os.path.exists(LBL) else []
done_i = {r["i"] for r in labels_done}

for i, r in replies.iterrows():          # يتخطى التقييمات المحفوظة
    if i in done_i:
        continue
    clear_output()
    print(f"[{i+1}/32] العميل:", r.text, "\n\nالسياق:\n", r.context, "\n\nالرد:", r.reply)
    while (kw := input("\nكلمة تدورينها في السياق (أو Enter): ").strip()):
        print("   ✅ موجودة" if kw.lower() in r.context.lower() else "   ❌ مو موجودة")
    a = int(input("1) يجاوب طلب العميل؟ (1/0): ")); b = int(input("2) كل معلوماته من السياق؟ (1/0): "))
    labels_done.append({"i": i, "answers": a, "grounded": b})
    pd.DataFrame(labels_done).to_csv(LBL, index=False)

manual = pd.read_csv(LBL).set_index("i")
print(f"يجاوب الطلب: {manual.answers.mean():.0%} | بدون اختراع: {manual.grounded.mean():.0%}")

يجاوب الطلب: 75% | بدون اختراع: 91%


### هل جودة البحث تأثر على جودة الرد؟
لكل تذكرة: كم وحدة من الثلاث سياقات اللي استلمها النموذج كانت من نفس نوع الطلب الصحيح؟

In [68]:
ev = replies.join(manual)
ev["n_relevant"] = [int((history.intent.values[top_idx(t)] == y).sum()) for t, y in zip(ev.text, ev.intent)]
ev.groupby("n_relevant")[["answers", "grounded"]].agg(["mean", "count"]).round(2)

answers       grounded      
              mean count     mean count
n_relevant                             
0             0.74    19     0.84    19
1             0.67     3     1.00     3
2             0.50     4     1.00     4
3             1.00     6     1.00     6

**النتيجة:** لما الثلاث سياقات كلها صحيحة، الرد يجاوب الطلب **100%** وبدون اختراع **100%**. كل حالات الاختراع صارت لما البحث فشل (صفر سياق صحيح). يعني **نموذج التوليد مو المشكلة، البحث هو الحلقة الضعيفة**، وهذا اللي دفعنا لتحسين البحث في القسم 5.

⚠️ مجموعات 1 و 2 صغيرة (3 و 4 تذاكر)، فالمقارنة الأوضح هي 0 مقابل 3.

## 8. الحكم الآلي (LLM-as-a-Judge) مقابل التقييم البشري
نموذج الحكم يقيّم نفس الـ 32 رد بنفس السؤالين، ونقيس اتفاقه معي بالنسبة و **Cohen's Kappa** (يشيل الاتفاق اللي يصير بالصدفة).

In [69]:
JUDGE_OUT = f"{PROJ}/judge_labels.csv"
JUDGE_PROMPT = """You are evaluating a customer-support reply. Answer with JSON only.

CUSTOMER MESSAGE (Gulf Arabic):
{text}

CONTEXT (past agent replies — the ONLY allowed source of information):
{context}

SYSTEM REPLY (Arabic):
{reply}

Question 1 "answers": Does the reply help the customer with their request?
- 1 if it answers the request, OR if the context has no answer and the reply appropriately hands off to a colleague.
- 0 if the reply is wrong or unrelated to the request, OR if the context contains a clear answer but the reply ignored it and only handed off.

Question 2 "grounded": Is every specific piece of information in the reply (steps, links, timeframes, page names, promises of actions) found in the context?
- 1 if all are in the context, OR if the reply has no specific information (only politeness and a hand-off).
- 0 if any specific information is NOT in the context.

Return exactly: {{"answers": 0 or 1, "grounded": 0 or 1, "reason": "short reason"}}"""

def judge(text, context, reply, tries=2):
    for _ in range(tries):
        r = client.chat.completions.create(model=JUDGE_MODEL, temperature=0, max_tokens=4000,
            messages=[{"role": "user", "content": JUDGE_PROMPT.format(text=text, context=context, reply=reply)}])
        m = re.search(r"\{[^{}]*\}", (r.choices[0].message.content if r.choices else "") or "", re.S)
        if m:
            try:
                d = json.loads(m.group(0)); return int(d["answers"]), int(d["grounded"]), str(d.get("reason", ""))
            except Exception:
                pass
        time.sleep(20)
    raise RuntimeError("الحكم ما رجّع JSON صالح")

jrows = pd.read_csv(JUDGE_OUT).to_dict("records") if os.path.exists(JUDGE_OUT) else []
jdone = {r["i"] for r in jrows}
for i, r in replies.iterrows():
    if i in jdone:
        continue
    try:
        a, g, reason = judge(r.text, r.context, r.reply)
    except Exception as e:
        print(f"❌ تذكرة {i+1} فشلت:", str(e)[:100]); continue
    jrows.append({"i": i, "j_answers": a, "j_grounded": g, "j_reason": reason})
    pd.DataFrame(jrows).to_csv(JUDGE_OUT, index=False)
    time.sleep(4)
print("أحكام محفوظة:", len(jrows), "من 32")

أحكام محفوظة: 32 من 32


In [70]:
cmp = ev.join(pd.read_csv(JUDGE_OUT).set_index("i"))
for h, j, name in [("answers", "j_answers", "يجاوب الطلب"), ("grounded", "j_grounded", "بدون اختراع")]:
    print(f"=== {name} === الاتفاق: {(cmp[h] == cmp[j]).mean():.0%} | Kappa: {cohen_kappa_score(cmp[h], cmp[j]):.2f} "
          f"| أنا نعم: {cmp[h].mean():.0%} | Dots نعم: {cmp[j].mean():.0%}")
    print(pd.crosstab(cmp[h], cmp[j], rownames=["أنا"], colnames=["Dots"]), "\n")
dis = cmp[(cmp.answers != cmp.j_answers) | (cmp.grounded != cmp.j_grounded)]
print("عدد التذاكر المختلف عليها:", len(dis))
dis[["text", "reply", "answers", "j_answers", "grounded", "j_grounded", "j_reason"]]

=== يجاوب الطلب === الاتفاق: 75% | Kappa: 0.00 | أنا نعم: 75% | Dots نعم: 100%
Dots   1
أنا     
0      8
1     24 

=== بدون اختراع === الاتفاق: 50% | Kappa: -0.05 | أنا نعم: 91% | Dots نعم: 53%
Dots   0   1
أنا         
0      1   2
1     14  15 

عدد التذاكر المختلف عليها: 20


,text,reply,answers,j_answers,grounded,j_grounded,j_reason
0,أبي أرجع الطلب واسترد فلوسى كيف الطريقة؟,أهلاً بك، سأطلب من أحد الزملاء متابعة طلبك لتوضيح خطوات الإرجاع والاسترداد بالتفصيل.,0,1,1,1,"The context does not provide clear steps for return and refund, so the reply appropriately hands off to a colleague ..."
1,المنتج وصلني خربان وأبي فلوسي ترجع لي بأسرع وقت!,أعتذر عن تجربتك السيئة، أحد الزملاء بيتابع معك بخصوص استرجاع المبلغ للمنتج التالف.,0,1,1,0,"The context contains no information about refunds or damaged products, so the reply appropriately hands off to a col..."
3,معليش ما ناسبني المنتج.ممكن أسترجعه واخذ المبلغ؟,أهلاً بك، أحد الزملاء بيتابع معك إجراءات الاسترجاع واسترداد المبلغ بالتفصيل.,0,1,1,1,Context has no return or refund information; reply appropriately hands off to a colleague.
5,تم قبول طلب الاسترجاع بس للحين المبلغ ما نزل بحسابي.,أعتذر عن التأخير في ظهور المبلغ، أحد الزملاء سيتابع معكم حالة التحويل البنكي للتأكد من وصوله لحسابكم في أقرب وقت.,0,1,1,0,"The context does not address the refund status, so hand-off is appropriate, but the reply includes specific details ..."
9,انخصم المبلغ من حسابي والطلب ما تأكد، وش الحل؟,نعتذر عن هذا الخطأ، يمكنك طلب استرداد المبلغ بالتواصل مع فريق خدمة العملاء وتزويدهم بتفاصيل العملية والمستندات الداع...,1,1,1,0,"The context lacks a specific solution for the deducted money issue, so the hand-off is appropriate, but the reply ad..."
10,السلام عليكم، مدى مو راضي يقبل الدفع عندكم، ممكن تساعدوني؟,"أهلاً بك، يمكنك مراجعة صفحة ""طرق الدفع"" على موقعنا للاطلاع على الخيارات المتاحة، وسيتواصل معك أحد الزملاء لتأكيد مدى...",0,1,1,0,"The context lacks information on مدي cards, so the hand-off is appropriate, but the reply introduces new details abo..."
12,أبي ألغي طلبي قبل لا ينشحن.,"أهلاً بك، يمكنك إلغاء طلبك قبل الشحن عبر تسجيل الدخول إلى حسابك ثم التوجه إلى صفحة ""إدارة الطلبات"" واختيار الطلب الم...",1,1,1,0,"The reply answers the cancellation request with steps from the context, but includes a promise about a colleague's a..."
13,خلاص ما عاد ابي الطلب الغوه الحين لو سمحتو,نعتذر عن أي إزعاج، يرجى تزويدنا برقم الطلب لنتمكن من مساعدتك في إلغاء طلبك.,0,1,1,1,"The reply appropriately requests the order number to proceed with cancellation, which is consistent with the context..."
14,طلبت مقاس غلط، اقدر ألغي الطلب وأطلب من جديد؟,أحد الزملاء بيتابع معك بخصوص إمكانية إلغاء الطلب وإعادة الطلب بالمقاس الصحيح.,1,1,1,0,"The context has no information on order cancellation, so the hand-off is appropriate, but the reply mentions a colle..."
15,ليش ما اقدر ألغي طلبي من التطبيق؟ أبي ألغيه!,أعتذر عن عدم تمكنك من إلغاء الطلب عبر التطبيق، أحد الزملاء بيتابع معك لتوضيح الخطوات اللازمة وإتمام الإلغاء إن أمكن.,1,1,0,1,"The context does not contain information about canceling an order from the app, so the reply appropriately hands off..."


| السؤال | الاتفاق | Kappa | أنا: نعم | Dots: نعم |
|---|---|---|---|---|
| يجاوب الطلب | 75% | 0.00 | 75% | 100% |
| بدون اختراع | 50% | -0.05 | 91% | 53% |

**الخلاصة:** Dots غير موثوق كحكم على هذي المهمة (الهدف ≥ 80% اتفاق ما تحقق).
1. **تساهل كامل** في السؤال الأول: قال "يجاوب" لكل الـ 32. نسبة الاتفاق (75%) خادعة، والـ Kappa = 0 يكشف ذلك.
2. **تناقض في تعليمات الحكم:** الـ prompt اعتبر "الوعود بإجراءات" معلومات محددة، وفي نفس الوقت قبل التحويل لزميل. فاعتبر Dots عبارة "أحد الزملاء بيتابع معك" اختراعاً في 14 تذكرة.

**الدروس:** نسبة الاتفاق لحالها مضللة والـ Kappa ضروري، وصياغة معايير الحكم تأثر بشكل كبير. لهذي المهمة التقييم البشري ضروري.

## 9. تحليل الأخطاء

In [71]:
P = clf.predict_proba(G5)
gulf["pred"], gulf["conf"] = clf.classes_[P.argmax(1)], P.max(1).round(2)
gulf["route"] = np.where(gulf.conf >= TAU, "auto", "human_review")
err = gulf[gulf.pred != gulf.intent][["text", "intent", "pred", "conf", "route"]]
print("أخطاء التصنيف:", len(err), "من", len(gulf), "| راحت للموظف (آمنة):", (err.route == "human_review").sum(),
      "| راحت تلقائي (خطيرة):", (err.route == "auto").sum())
err

أخطاء التصنيف: 9 من 40 | راحت للموظف (آمنة): 8 | راحت تلقائي (خطيرة): 1


,text,intent,pred,conf,route
1,المنتج وصلني خربان وأبي فلوسي ترجع لي بأسرع وقت!,get_refund,delivery_period,0.14,human_review
12,السلام عليكم، مدى مو راضي يقبل الدفع عندكم، ممكن تساعدوني؟,payment_issue,get_refund,0.22,human_review
13,ليه كل ما جيت ادفع يرفض العمليه مع ان رصيدي موجود؟!,payment_issue,track_refund,0.23,human_review
17,طلبت مقاس غلط، اقدر ألغي الطلب وأطلب من جديد؟,cancel_order,change_shipping_address,0.16,human_review
21,ممكن تعطوني تحديث عن حالة الشحن؟,track_order,track_refund,0.22,human_review
24,مكتوب ان الطلب تم شحنه من يومين، بس ما فيه اي تحديث جديد.,track_order,track_refund,0.27,human_review
28,ما توصلني رسالة اعادة تعيين الرقم السري على الايميل، ليه؟,recover_password,change_shipping_address,0.36,auto
36,الصراحة تجربتي معكم كانت سيئة والمنتج جودته أقل بكثير من المتوقع.,complaint,track_refund,0.17,human_review
39,والله طفشت من سوء التعاملدفعت فلوسي وبالأخير هذي الخدمة اللي أحصل عليها؟!,complaint,track_refund,0.20,human_review


| # | الرسالة | الصحيح | توقع النظام | الثقة | المسار | السبب |
|---|---|---|---|---|---|---|
| 1 | المنتج وصلني خربان وأبي فلوسي ترجع | get_refund | delivery_period | 0.14 | موظف | غموض: شكوى + طلب استرداد، و"وصلني" سحبتها للتوصيل |
| 2 | مدى مو راضي يقبل الدفع | payment_issue | get_refund | 0.22 | موظف | نقص تغطية: "مدى" مصطلح محلي |
| 3 | ليه كل ما جيت ادفع يرفض العملية مع ان رصيدي موجود | payment_issue | track_refund | 0.23 | موظف | انحياز لكلمات الفلوس ("رصيدي") |
| 4 | طلبت مقاس غلط، اقدر الغي الطلب واطلب من جديد؟ | cancel_order | change_shipping_address | 0.16 | موظف | غموض: طلبين في رسالة وحدة |
| 5 | ممكن تعطوني تحديث عن حالة الشحن؟ | track_order | track_refund | 0.22 | موظف | خلط بين نوعين متشابهين |
| 6 | مكتوب ان الطلب تم شحنه من يومين بس ما فيه تحديث | track_order | track_refund | 0.27 | موظف | خلط بين نوعين متشابهين |
| 7 | ما توصلني رسالة اعادة تعيين الرقم السري على الايميل | recover_password | change_shipping_address | 0.36 | **تلقائي** ⚠️ | خطأ خطير: "الرقم السري" (خليجي) بدل "كلمة المرور" |
| 8 | الصراحة تجربتي معكم كانت سيئة والمنتج جودته أقل | complaint | track_refund | 0.17 | موظف | نقص تغطية: أسلوب الشكوى العامة |
| 9 | والله طفشت من سوء التعامل، دفعت فلوسي... | complaint | track_refund | 0.20 | موظف | انحياز لكلمات الفلوس ("فلوسي") |
| 10 | (نفس #7 في مرحلة التوليد) | recover_password | change_shipping_address | | | خطأ توليد ناتج عن التصنيف: الرد كشف للعميل التصنيف الداخلي |

**الأنماط:** انحياز لـ `track_refund` (4 من 9)، خلط بين الأنواع المتشابهة، مصطلحات محلية غير مغطاة. الحد τ صاد **8 من 9** أخطاء قبل ما توصل للعميل.

**تحسينات مقترحة:** أمثلة تدريب بمصطلحات محلية ("مدى"، "الرقم السري")، وشكاوى فيها كلمات فلوس، ومنع الرد من ذكر نوع الطلب الداخلي.

## 10. النظام الكامل: `assist()` + Demo
أداة مساعدة **لموظف خدمة العملاء**: تصنف التذكرة، وتقرر هل تنحل تلقائياً أو تتحول، وتستخرج الحقول، وتقترح رداً يراجعه الموظف.

1. **التصنيف:** e5 + Logistic Regression (مدرّب على الإنجليزي + الخليجي).
2. **البحث المحسّن:** أقرب 3 ردود موظفين من نفس النوع المتوقع.
3. **الحقول بأرخص أداة:** رقم الطلب بـ Regex، والأولوية بقواعد.
4. **الرد:** Nemotron يكتب رداً عربياً من السياق. لو فشل أو كان مطفي، رد عربي ثابت (بدون كشف بيانات عملاء آخرين).

In [72]:
ORDER_RE  = re.compile(r"\b[A-Z]-\d{4,6}\b|[0-9٠-٩]{4,}")
URGENT_RE = re.compile(r"عاجل|بسرعة|فورا|فوراً|الحين|تكفون|!!|urgent|asap", re.I)
HIGH = {"payment_issue", "complaint", "get_refund", "track_refund"}
FALLBACK = "شكراً لتواصلك معنا، تم استلام طلبك وبيتواصل معك أحد الزملاء بأقرب وقت."

def assist(text, use_llm=True):
    q = U.embed([text], prefix="query: ")[0]
    proba = clf.predict_proba(q[None])[0]
    k = proba.argmax()
    intent, conf = clf.classes_[k], float(proba[k])
    sims = np.where(history.intent.values == intent, hist_emb @ q, -1)
    top = np.argsort(-sims)[:3]
    order = ORDER_RE.search(text)
    priority = "high" if intent in HIGH else "medium"
    if URGENT_RE.search(text):
        priority = "urgent" if priority == "high" else "high"
    reply, source = FALLBACK, "رد ثابت (احتياطي)"
    if use_llm:
        try:
            reply, source = generate(text, intent, build_context(top)), GEN_MODEL
        except Exception as e:
            source = f"رد ثابت (الـ LLM ما رد: {str(e)[:60]})"
    return {"intent": intent, "confidence": round(conf, 3),
            "route": "auto" if conf >= TAU else "human_review", "priority": priority,
            "order_id": order.group(0) if order else None,
            "similar_tickets": history.ticket_id.values[top].tolist(),
            "reply": reply, "reply_source": source,
            "agent_reference": history.response[top[0]][:300]}

assist("وين طلبي رقم 48213؟ صار له اسبوع تكفون بسرعة", use_llm=False)

{'intent': 'track_order',
 'confidence': 0.296,
 'route': 'human_review',
 'priority': 'high',
 'order_id': '48213',
 'similar_tickets': [23049, 22968, 23055],
 'reply': 'شكراً لتواصلك معنا، تم استلام طلبك وبيتواصل معك أحد الزملاء بأقرب وقت.',
 'reply_source': 'رد ثابت (احتياطي)',
 'agent_reference': 'Grateful for your contact! I get the sense that you need assistance with locating your order with the order number A-67984. To do so, please follow these steps:\n\n1. Visit our website at demomart.example.\n2. Log in to your account using your credentials.\n3. Navigate to the "Order History" section.\n4.'}

In [73]:
import inspect
try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    import gradio as gr

AR = {"get_refund": "💸 استرجاع مبلغ", "track_refund": "🔎 متابعة استرجاع",
      "payment_issue": "💳 مشكلة دفع", "cancel_order": "✖️ إلغاء طلب",
      "track_order": "📦 تتبع طلب", "recover_password": "🔑 استعادة كلمة المرور",
      "change_shipping_address": "🏠 تغيير العنوان", "complaint": "😔 شكوى"}
PR = {"urgent": "🔴 عاجلة", "high": "🟠 عالية", "medium": "🟢 عادية"}

def run(text, use_llm):
    if not text.strip():
        return "", "", "", "", "", "اكتبي رسالة العميلة أول 🌸", {}
    o = assist(text, use_llm=use_llm)
    route = "✨ تلقائي" if o["route"] == "auto" else "🙋‍♀️ تحويل لموظفة"
    return (AR.get(o["intent"], o["intent"]), f"{o['confidence']:.0%}", route,
            PR[o["priority"]], o["order_id"] or "—", o["reply"], o)

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&display=swap');
.gradio-container {background: linear-gradient(160deg,#fff0f6 0%,#fde2ef 45%,#ede4ff 100%) !important;
                   font-family:'Tajawal',sans-serif !important; direction:rtl;}
.gradio-container * {font-family:'Tajawal',sans-serif !important;}
#hero {text-align:center; padding:28px 12px 8px;}
#hero h1 {font-size:2.4rem; font-weight:800; color:#d6336c; margin:0;}
#hero p {font-size:1.1rem; color:#9c6ade; margin:6px 0 0;}
.card {background:#ffffffcc !important; border:2px solid #ffd6e7 !important;
       border-radius:22px !important; box-shadow:0 6px 20px #f7a8c933 !important;}
#send {background:linear-gradient(90deg,#ff8fb8,#c79bff) !important; color:white !important;
       border:none !important; border-radius:999px !important; font-size:1.15rem !important; font-weight:700 !important;}
#reply textarea {font-size:1.15rem !important; line-height:1.9 !important; background:#fff7fb !important; color:#4a2040 !important;}
.chip textarea, .chip input {text-align:center !important; font-weight:700 !important; color:#b0306a !important;
       background:#fff0f6 !important; border-radius:16px !important;}
label span {color:#a8467a !important; font-weight:700 !important;}
footer {display:none !important;}
"""
FORCE_LIGHT = """() => { const u = new URL(window.location);
  if (u.searchParams.get('__theme') !== 'light') { u.searchParams.set('__theme','light'); window.location.href = u.href; } }"""
THEME = gr.themes.Soft(primary_hue="pink", secondary_hue="purple", radius_size="lg")

bk, lk = {"title": ""}, {"share": True}
if "css" in inspect.signature(gr.Blocks.__init__).parameters:
    bk.update(theme=THEME, css=CSS, js=FORCE_LIGHT)
else:
    lk.update(theme=THEME, css=CSS, js=FORCE_LIGHT)

with gr.Blocks(**bk) as demo:
    gr.HTML("<div id='hero'><h1>🦌ريم </h1><p>مساعد خدمة العملاء   💗</p></div>")
    with gr.Group(elem_classes="card"):
        msg = gr.Textbox(lines=3, label="💌 رسالة العميلة", placeholder="مثلاً: البجامة وصلتني مقاس غلط وابي ارجعها...")
        send = gr.Button("إرسال 💗", elem_id="send")
    with gr.Group(elem_classes="card"):
        reply = gr.Textbox(lines=5, label="💬 الرد المقترح", elem_id="reply")
    with gr.Row():
        intent = gr.Textbox(label="نوع الطلب", elem_classes="chip")
        conf = gr.Textbox(label="الثقة", elem_classes="chip")
        route = gr.Textbox(label="المسار", elem_classes="chip")
        prio = gr.Textbox(label="الأولوية", elem_classes="chip")
        order = gr.Textbox(label="رقم الطلب", elem_classes="chip")
    with gr.Accordion("🔍 تفاصيل للموظفة", open=False):
        full = gr.JSON(label="المخرج الكامل")
    with gr.Accordion("⚙️ إعدادات", open=False):
        use_llm = gr.Checkbox(value=True, label="كتابة الرد بالذكاء الاصطناعي (كل رسالة = طلب API)")
    gr.Examples([["ودي ارجع البجامة وآخذ فلوسي"], ["الفلوس اللي رجعتوها للحين ما وصلت حسابي"],
                 ["طلبي رقم 55821 ما وصل تكفون بسرعة!!"], ["نسيت الرقم السري وما اقدر ادخل"]],
                inputs=[msg], label="🌸 أمثلة")
    outs = [intent, conf, route, prio, order, reply, full]
    send.click(run, [msg, use_llm], outs)
    msg.submit(run, [msg, use_llm], outs)

demo.launch(**lk)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff1e18f861c61496ab.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [74]:
import inspect
try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    import gradio as gr

AR = {"get_refund": "💸 استرجاع مبلغ", "track_refund": "🔎 متابعة استرجاع",
      "payment_issue": "💳 مشكلة دفع", "cancel_order": "✖️ إلغاء طلب",
      "track_order": "📦 تتبع طلب", "recover_password": "🔑 استعادة كلمة المرور",
      "change_shipping_address": "🏠 تغيير العنوان", "complaint": "😔 شكوى"}
PR = {"urgent": "🔴 عاجلة", "high": "🟠 عالية", "medium": "🟢 عادية"}

def run(text, use_llm):
    if not text.strip():
        return "", "", "", "", "", "اكتبي رسالة العميلة أول 🌸", {}
    o = assist(text, use_llm=use_llm)
    route = "✨ تلقائي" if o["route"] == "auto" else "🙋‍♀️ تحويل لموظفة"
    return (AR.get(o["intent"], o["intent"]), f"{o['confidence']:.0%}", route,
            PR[o["priority"]], o["order_id"] or "—", o["reply"], o)

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&display=swap');
.gradio-container {background: linear-gradient(160deg,#fff0f6 0%,#fde2ef 45%,#ede4ff 100%) !important;
                   font-family:'Tajawal',sans-serif !important; direction:rtl;}
.gradio-container * {font-family:'Tajawal',sans-serif !important;}
#hero {text-align:center; padding:28px 12px 8px;}
#hero h1 {font-size:2.4rem; font-weight:800; color:#d6336c; margin:0;}
#hero p {font-size:1.1rem; color:#9c6ade; margin:6px 0 0;}
.card {background:#ffffffcc !important; border:2px solid #ffd6e7 !important;
       border-radius:22px !important; box-shadow:0 6px 20px #f7a8c933 !important;}
#send {background:linear-gradient(90deg,#ff8fb8,#c79bff) !important; color:white !important;
       border:none !important; border-radius:999px !important; font-size:1.15rem !important; font-weight:700 !important;}
#reply textarea {font-size:1.15rem !important; line-height:1.9 !important; background:#fff7fb !important; color:#4a2040 !important;}
.chip textarea, .chip input {text-align:center !important; font-weight:700 !important; color:#b0306a !important;
       background:#fff0f6 !important; border-radius:16px !important;}
label span {color:#a8467a !important; font-weight:700 !important;}
footer {display:none !important;}
"""
FORCE_LIGHT = """() => { const u = new URL(window.location);
  if (u.searchParams.get('__theme') !== 'light') { u.searchParams.set('__theme','light'); window.location.href = u.href; } }"""
THEME = gr.themes.Soft(primary_hue="pink", secondary_hue="purple", radius_size="lg")

bk, lk = {"title": "ريم 🦌"}, {"share": True}
if "css" in inspect.signature(gr.Blocks.__init__).parameters:
    bk.update(theme=THEME, css=CSS, js=FORCE_LIGHT)
else:
    lk.update(theme=THEME, css=CSS, js=FORCE_LIGHT)

with gr.Blocks(**bk) as demo:
    gr.HTML("<div id='hero'><h1>ريم 🦌</h1><P>مساعد خدمة العملاء   💗</p></div>")
    with gr.Group(elem_classes="card"):
        msg = gr.Textbox(lines=3, label="💌 رسالة العميلة", placeholder="مثلاً: البجامة وصلتني مقاس غلط وابي ارجعها...")
        send = gr.Button("إرسال 💗", elem_id="send")
    with gr.Group(elem_classes="card"):
        reply = gr.Textbox(lines=5, label="💬 الرد المقترح", elem_id="reply")
    with gr.Row():
        intent = gr.Textbox(label="نوع الطلب", elem_classes="chip")
        conf = gr.Textbox(label="الثقة", elem_classes="chip")
        route = gr.Textbox(label="المسار", elem_classes="chip")
        prio = gr.Textbox(label="الأولوية", elem_classes="chip")
        order = gr.Textbox(label="رقم الطلب", elem_classes="chip")
    with gr.Accordion("🔍 تفاصيل للموظفة", open=False):
        full = gr.JSON(label="المخرج الكامل")
    with gr.Accordion("⚙️ إعدادات", open=False):
        use_llm = gr.Checkbox(value=True, label="كتابة الرد بالذكاء الاصطناعي (كل رسالة = طلب API)")
    gr.Examples([["ودي ارجع البجامة وآخذ فلوسي"], ["الفلوس اللي رجعتوها للحين ما وصلت حسابي"],
                 ["طلبي رقم 55821 ما وصل تكفون بسرعة!!"], ["نسيت الرقم السري وما اقدر ادخل"]],
                inputs=[msg], label="🌸 أمثلة")
    outs = [intent, conf, route, prio, order, reply, full]
    send.click(run, [msg, use_llm], outs)
    msg.submit(run, [msg, use_llm], outs)

demo.launch(**lk)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://01f581d7369f735d99.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. الخلاصة

| | النتيجة |
|---|---|
| التصنيف على الخليجي (Macro-F1) | 0.40 ← **0.79** (والإنجليزي ثابت 0.99) |
| البحث: الثلاث سياقات صحيحة | 18% ← **78%** |
| الردود تجاوب الطلب / بدون اختراع (تقييم بشري) | 75% / 91% |
| الحكم الآلي مقابل البشري (Kappa) | 0.00 و -0.05 ← غير موثوق |
| الأخطاء اللي صادها الحد τ | 8 من 9 |

**الخطوات القادمة:** Fine-tuning للمصنّف بالبيانات الخليجية، إضافة مقالات سياسات المتجر للبحث، تصحيح تعليمات الحكم الآلي، وتوسيع الاختبار الخليجي.